In [1]:
from sklearn.metrics import pairwise_distances

# Setup

%load_ext autoreload
%autoreload 2

import os

import torch
import numpy as np
from umap import UMAP
from dotenv import load_dotenv
from pandas import read_json
from sklearn.cluster import HDBSCAN
from transformers import AutoModel, AutoTokenizer

from pipelines.multi_service.pooling_layer import mean_pooling

load_dotenv()

print(f"HuggingFace cache directory is {os.environ.get('HF_HOME')}")


def should_be_skipped(class_name: str):
    return (
        class_name.endswith("Test")
        or class_name.endswith("Tests")
        or class_name.endswith("IT")
    )

HuggingFace cache directory is /media/martin/BigData/datasets/cache/huggingface/


In [ ]:
import pandas as pd

GENERIC_WORDS = [
    "get",
    "set",
    "find by",
    "exists by",
    "exists",
    "update",
    "delete",
    "delete by",
    "create",
    "get all",
    "insert",
    "insert into",
    "find all",
    "find all by",
    "remove",
    "add",
    "edit",
    "save",
    "list",
    "view",
]
GENERIC_WORDS.sort(key=len, reverse=True)

MODEL = "microsoft/codebert-base"

codeBertTokenizer = AutoTokenizer.from_pretrained(MODEL)
codeBertModel = AutoModel.from_pretrained(MODEL)

input_file = input("Enter input file path: ")
output_file = input("Enter output file path: ")

data_frame = read_json(input_file, lines=True)


def extract_property(methods: list[dict[str, str]], property: str):
    return list(map(lambda m: m[property], methods))


def remove_words(text: str):
    for word in GENERIC_WORDS:
        if text.startswith(word):
            text = text.removeprefix(word).strip()

    return text


def evaluate(inputs: list[dict[str, str]], _fully_qualified_name: str):
    signature = extract_property(inputs, "signature")
    mean_pooled_signature = get_embeddings(signature)
    mean_pooled_numpy = mean_pooled_signature.cpu().numpy()

    umap = UMAP(
        n_neighbors=5,
        min_dist=0.0,
        n_components=15,
        metric="euclidean",
        random_state=42,
        init="random",
    )
    umap_reduced = umap.fit_transform(mean_pooled_numpy)
    clustering_alg = HDBSCAN(
        min_samples=1,
        min_cluster_size=3,
        metric="cosine",
        allow_single_cluster=True,
        store_centers="medoid",
    )

    labels = clustering_alg.fit_predict(umap_reduced)

    unique_labels = set(labels)
    number_of_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    noise_count = list(labels).count(-1)
    noise_ratio = noise_count / len(labels)

    print(labels)
    print(clustering_alg.medoids_)
    medoids = clustering_alg.medoids_
    unique_clusters = unique_labels - {-1}
    avg_distance = 0.0
    if len(unique_clusters) > 1:
        if len(unique_clusters) > 1:
            distances = pairwise_distances(medoids, metric="cosine")
            print(distances)
            tri_offset = np.triu_indices(distances.shape[0], k=1)
            unique_distances = distances[tri_offset]

            avg_distance = np.mean(unique_distances)

    if all(x == -1 for x in labels):
        number_of_clusters = -1

    return (
        number_of_clusters,
        noise_ratio,
        labels,
        avg_distance,
    )


def get_embeddings(inputs, tokenizer=codeBertTokenizer, model=codeBertModel):
    inputs = list(map(lambda input: "<decoder-only> " + input, inputs))

    tokenized_inputs = tokenizer(
        inputs, return_tensors="pt", padding=True, truncation=True, max_length=512
    )
    with torch.no_grad():
        output = model(**tokenized_inputs)
    attention_mask = tokenized_inputs["attention_mask"]
    return mean_pooling(output, attention_mask)


data_frame["optimal_k"] = pd.Series()
data_frame["noise_ratio"] = pd.Series()
data_frame["cluster"] = pd.Series()
data_frame["avg_dist"] = pd.Series(dtype="float64")
results = []
for index, row in data_frame.iterrows():
    class_name = row["fullyQualifiedName"]
    methods = row["methods"]

    if len(methods) < 10:
        print(f"Skipped small class {class_name}, number of methods {len(methods)}")
        continue

    if should_be_skipped(class_name):
        print(f"Skipped Test class {class_name}, number of methods {len(methods)}")
        continue

    optimal_k, noise_ratio, labels, avg_dist = evaluate(methods, class_name)

    data_frame.loc[index, "optimal_k"] = optimal_k
    data_frame.loc[index, "avg_dist"] = avg_dist * 100
    data_frame.loc[index, "noise_ratio"] = noise_ratio
    data_frame.at[index, "cluster"] = labels

    results.append(f"{class_name}: {optimal_k}, {labels}, dist: {avg_dist:.4f}")

for result in results:
    print(result)

data_frame.to_json(output_file, lines=True, orient="records")

In [ ]:
# Evaluation of results

import pandas as pd

input_file = input("Enter input file path: ")

df = pd.read_json(input_file, lines=True)
df = df.query("optimal_k.notnull()")
more_cluster_lines = df.query("optimal_k > 2")
one_cluster_lines = df.drop(more_cluster_lines.index)


def count_rows_and_print(df: pd.DataFrame, query: str, stat_type: str):
    df_part = df.query(query)
    count = len(df_part)
    print(f"{stat_type}: {count}")
    for _, record in df_part.iterrows():
        print(record["fullyQualifiedName"])
    print("---")
    return count


false_negatives = count_rows_and_print(
    one_cluster_lines, "label == 1", "False negatives"
)
true_negatives = count_rows_and_print(one_cluster_lines, "label == 0", "True negatives")
false_positives = count_rows_and_print(
    more_cluster_lines, "label == 0", "False positives"
)
true_positives = count_rows_and_print(
    more_cluster_lines, "label == 1", "True positives"
)

precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)
accuracy = (true_positives + true_negatives) / (
    true_positives + false_positives + false_negatives + true_negatives
)
f1_score = 2 * ((precision * recall) / (precision + recall))

print(f"""
Stats
Precision: {precision}
Recall: {recall}
Accuracy: {accuracy}
F1 score: {f1_score}
-------------
""")

df

In [ ]:
from pipelines.shared import input_until_integer

# Labeling script

input_file = input("Enter input file path: ")
output_file = input("Enter output file path: ")

df = read_json(input_file, lines=True)

df["label"] = pd.Series()

for index, row in df.iterrows():
    name = row["fullyQualifiedName"]
    methods = row["methods"]
    assigned_label = row["label"]
    lcom4 = row["lcom4"]

    label = 0
    if not should_be_skipped(name) and len(methods) > 10:
        print("-" * 10)
        print(f"Class {name} (LCOM4 {lcom4})")
        print("Methods: ")
        for method in methods:
            print(method)

        label = input_until_integer("Enter label: ")

    df.loc[index, "label"] = label
    df.to_json(output_file, lines=True, orient="records")